# CRE — whole-system first light on real papers

**Whole-system first light. It is not a precision estimate and it does not use the production launcher.** The saved manifest determines its own reportability; the notebook does not predeclare that result.

This notebook retrieves real PMC JATS XML, runs both installed bands, joins them through the canonical disposition artifact, and then answers two separate questions for every taxonomy stratum:

1. Did the check actually run on these papers?
2. If it ran, what did it find?

A zero is never printed without its reachability state.

## Pipeline blueprint

`NCBI PMC XML → parser → Band 1 (F1/F2/F8) → canonical preband disposition → Band 2 (F3/F4/F6) → artifact-derived F1–F8 report`

F5 and F7 are shown as **NOT RUN** on real data because this checkout has no production evidence builder for either path. The notebook does not substitute fixtures or pretend their zero is clean.

## Safety and cost

- Cells 1–4 are free: checkout, tests, real-paper retrieval, parsing, and transport conformance.
- Cell 5 is a hard cost gate.
- Cells 6–7 make Anthropic calls.
- Every run writes to a new timestamped Drive directory. No Drive artifact is deleted or overwritten.
- Restart the Colab session before Cell 1 so stale `cre.f1` modules cannot survive a checkout change.


## Cell 1 — bootstrap and provenance

Fresh pinned checkout, Drive mount, and exact code identity.


In [ ]:
# Estimated runtime: ~45–90 s (pinned packages + clone/fetch + Drive mount)
import os, sys, json, re, time, hashlib, tempfile, subprocess, importlib
from pathlib import Path
from datetime import datetime, timezone

BRANCH = "merge/f2-into-f3f7"
EXPECTED_COMMIT = "0a8e663b926af94f7da8d67964118775ab580268"
EXPECTED_BAND_PROMPTS_BLOB = "fa01126e2b9482d450065fd70cd0eb1fea816f5c"
REPO_URL = "https://github.com/astonliu/citation-repair-engine.git"
REPO = Path("/content/cre-whole-system")
PKG_ROOT = REPO / "citation_repair_F1_handoff"
F1_DIR = PKG_ROOT / "cre" / "f1"
EMAIL = "aston.hliu@gmail.com"
MODEL = "claude-opus-5"

subprocess.run([
    sys.executable, "-m", "pip", "-q", "install",
    "rapidfuzz==3.14.5", "requests==2.32.5", "lxml==6.0.2", "anthropic==0.122.0",
    "jsonschema==4.26.0", "pytest==9.1.1"
], check=True)

from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

def get_secret(name, required=False):
    try:
        value = userdata.get(name) or ""
    except Exception:
        value = ""
    if required and not value:
        raise RuntimeError(f"{name} is missing from Colab Secrets")
    return value

if any(name == "cre" or name.startswith("cre.f1") for name in sys.modules):
    raise RuntimeError("Stale CRE modules are already loaded. Restart the Colab session, then rerun Cell 1.")

if REPO.exists():
    assert (REPO / ".git").exists(), f"{REPO} exists but is not a git checkout"
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    assert not dirty, f"Refusing to reset a dirty Colab checkout:\n{dirty}"
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
remote_commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", f"origin/{BRANCH}"], text=True).strip()
assert remote_commit == EXPECTED_COMMIT, (remote_commit, EXPECTED_COMMIT)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", remote_commit], check=True)

CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
STATUS = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
BLOB = subprocess.check_output([
    "git", "-C", str(REPO), "rev-parse",
    "HEAD:citation_repair_F1_handoff/cre/f1/band_prompts.py"
], text=True).strip()
assert CODE_COMMIT == EXPECTED_COMMIT, (CODE_COMMIT, EXPECTED_COMMIT)
assert STATUS == "", STATUS
assert BLOB == EXPECTED_BAND_PROMPTS_BLOB, (BLOB, EXPECTED_BAND_PROMPTS_BLOB)

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
importlib.invalidate_caches()

from cre.f1 import (band_prompts, evidence_reader, eval_report, fulltext_reader,
                    judgment_run, ncbi_meta, parser, preband_contract,
                    preband_disposition, production_launcher, ratelimit)
from cre.f1.recording_adapter import AdapterReceipt, wrap_run_seams

assert len(production_launcher.GOVERNING_MODULES) == 17
RUN_ID = globals().get("RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
DATA = Path("/content/drive/MyDrive/Citation-Integrity/Data")
RUN_ROOT = DATA / "whole_system_first_light" / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("HEAD:", CODE_COMMIT)
print("git status: CLEAN")
print("band_prompts.py blob:", BLOB)
print("governing modules:", len(production_launcher.GOVERNING_MODULES))
print("run directory:", RUN_ROOT)
print("BOOTSTRAP: PASS")


## Cell 2 — deterministic offline suite

The optional MedCPT cache and network are isolated so the suite tests the pinned environment, not Colab luck.


In [ ]:
# Estimated runtime: ~80–100 s (full offline suite; MedCPT network/cache isolated)
import importlib.metadata as metadata

offline_cache = Path(tempfile.mkdtemp(prefix="cre_hf_offline_empty_"))
env = os.environ.copy()
env.update({
    "PYTHONPATH": ".",
    "PYTHONDONTWRITEBYTECODE": "1",
    "HF_HOME": str(offline_cache),
    "TRANSFORMERS_CACHE": str(offline_cache / "transformers"),
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "cre/f1", "-q"],
    cwd=PKG_ROOT, env=env, text=True, capture_output=True
)
output = proc.stdout + "\n" + proc.stderr
print("\n".join(output.splitlines()[-15:]))
assert proc.returncode == 0, f"offline pytest failed with exit code {proc.returncode}"

def count(kind):
    hit = re.search(rf"(\d+)\s+{kind}", output)
    return int(hit.group(1)) if hit else 0

suite = {"passed": count("passed"), "skipped": count("skipped"), "xfailed": count("xfailed")}
versions = {name: metadata.version(name) for name in ("rapidfuzz", "requests", "lxml", "anthropic", "jsonschema", "pytest")}
print("environment:", json.dumps(versions, sort_keys=True))
print("suite:", json.dumps(suite, sort_keys=True))
assert suite == {"passed": 2480, "skipped": 12, "xfailed": 39}, suite
print("OFFLINE SUITE: PASS")


## Cell 3 — retrieve and parse real papers

Downloads three real PMC JATS articles. The default active corpus is the 289-reference paper so the whole-system run exercises a substantial bibliography rather than a tiny smoke-test document.


In [ ]:
# Estimated runtime: ~10–25 s (three live PMC EFetch downloads + local parsing)
import shutil
import xml.etree.ElementTree as ET
import requests

# Three real papers are retrieved. The 289-reference article is the default
# active corpus; the smaller papers remain available for optional smoke tests.
DOWNLOAD_PMCIDS = ["PMC13294766", "PMC13294470", "PMC13295119"]
ACTIVE_PMCIDS = ["PMC13295119"]
MIN_ACTIVE_REFERENCES = 100
assert set(ACTIVE_PMCIDS) <= set(DOWNLOAD_PMCIDS)

NCBI_API_KEY = get_secret("NCBI_API_KEY")
ratelimit.configure_ncbi(bool(NCBI_API_KEY))
DOWNLOAD_DIR = RUN_ROOT / "retrieved_papers"
XML_DIR = RUN_ROOT / "active_corpus"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
XML_DIR.mkdir(parents=True, exist_ok=True)
http = requests.Session()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def retrieve_pmc_xml(pmcid):
    assert re.fullmatch(r"PMC\d+", pmcid), pmcid
    destination = DOWNLOAD_DIR / f"{pmcid}.xml"
    if destination.exists():
        payload = destination.read_bytes()
        source = "existing durable download"
    else:
        params = {
            "db": "pmc", "id": pmcid[3:], "retmode": "xml",
            "tool": "CRE_whole_system_first_light", "email": EMAIL,
        }
        if NCBI_API_KEY:
            params["api_key"] = NCBI_API_KEY
        response = http.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params=params, timeout=90
        )
        response.raise_for_status()
        payload = response.content
        destination.write_bytes(payload)
        source = "live NCBI PMC EFetch"
    root = ET.fromstring(payload)
    articles = [node for node in root.iter() if node.tag.rsplit("}", 1)[-1] == "article"]
    assert root.tag.rsplit("}", 1)[-1] == "article" or articles, \
        f"{pmcid}: response is XML but contains no JATS article"
    title_nodes = [node for node in root.iter() if node.tag.rsplit("}", 1)[-1] == "article-title"]
    title = " ".join("".join(title_nodes[0].itertext()).split()) if title_nodes else "TITLE NOT FOUND"
    assert len(payload) > 10000, f"{pmcid}: implausibly small XML response ({len(payload)} bytes)"
    return destination, payload, source, title

retrieved = {}
for pmcid in DOWNLOAD_PMCIDS:
    path, payload, source, title = retrieve_pmc_xml(pmcid)
    retrieved[pmcid] = path
    print(f"{pmcid}: {len(payload):,} bytes | {sha256_bytes(payload)[:16]} | {source}")
    print("  title:", title[:240])

for pmcid in ACTIVE_PMCIDS:
    src = retrieved[pmcid]
    dst = XML_DIR / src.name
    if dst.exists():
        assert dst.read_bytes() == src.read_bytes(), f"refusing to overwrite changed {dst}"
    else:
        shutil.copy2(src, dst)

CORPUS_MANIFEST_PATH = RUN_ROOT / "active_corpus_manifest.json"
if CORPUS_MANIFEST_PATH.exists():
    corpus_manifest = json.loads(CORPUS_MANIFEST_PATH.read_text(encoding="utf-8"))
    inventory = preband_contract.corpus_inventory(corpus_manifest)
    preband_contract.verify_corpus_contents(str(XML_DIR), inventory)
else:
    corpus_manifest = preband_contract.build_corpus_manifest(
        str(XML_DIR), str(CORPUS_MANIFEST_PATH)
    )

ACTIVE_REFERENCE_COUNTS = {}
for path in sorted(XML_DIR.glob("*.xml")):
    refs = parser.parse_pmc_xml(str(path))
    assert refs, f"{path.name}: parser found zero references"
    failures = sum(len(r.citance_sentence_partition_failures) for r in refs)
    ACTIVE_REFERENCE_COUNTS[path.stem] = len(refs)
    print(f"{path.name}: {len(refs)} references | sentence partition events: {failures}")

active_reference_total = sum(ACTIVE_REFERENCE_COUNTS.values())
assert active_reference_total >= MIN_ACTIVE_REFERENCES, (
    f"Active corpus is too small for the whole-system test: "
    f"{active_reference_total} references; require at least {MIN_ACTIVE_REFERENCES}"
)
print(f"active bibliography size gate: PASS ({active_reference_total} references)")

print("active corpus manifest:", CORPUS_MANIFEST_PATH)
print("REAL-PAPER RETRIEVAL AND PARSE: PASS")


## Cell 4 — F1 transport conformance

Runs controlled failure and genuine-absence responses through the real Band-1 orchestration and serializers.


In [ ]:
# Estimated runtime: ~2 s (controlled transport fixtures through the real pipeline; no network/model)
from unittest.mock import patch
import requests
from cre.f1 import confirm, lookup
from cre.f1 import run as band1_run
from cre.f1.schema import (ClaimedRef, Reference, F1,
                           FETCH_ANSWERED_ABSENT, FETCH_RESOLVER_ERROR)

class FakeResponse:
    def __init__(self, status_code=200, text="", json_data=None, headers=None):
        self.status_code = status_code
        self.text = text
        self._json = json_data
        self.headers = headers or {}
    def json(self):
        if self._json is None:
            raise ValueError("no JSON body")
        return self._json

class FakeSession:
    def __init__(self, handler):
        self.handler = handler
    def get(self, url, params=None, timeout=None):
        return self.handler(url, params)

def ref(cid, pmid):
    return Reference(cid, "A controlled citance [1].", ClaimedRef(
        title="A plausible sounding study of nothing",
        authors=["Smith J"], year=2020, claimed_pmid=pmid))

def transport_failure(_url, _params):
    raise requests.ConnectionError("deliberate transport failure")

def healthy_absence(url, _params):
    if url == lookup.EFETCH:
        return FakeResponse(status_code=200, text="")
    if url == confirm.PUBMED_ESEARCH:
        return FakeResponse(json_data={"esearchresult": {"idlist": []}})
    if url == confirm.CROSSREF_URL:
        return FakeResponse(json_data={"status": "ok", "message": {"items": []}})
    if url == confirm.OPENALEX_URL:
        return FakeResponse(json_data={"results": []})
    raise AssertionError(f"unexpected URL: {url}")

def must_not_call(_prompt):
    raise AssertionError("transport failure should short-circuit before the model")

def fabrication_fixture(_prompt):
    return '{"verdict":"fabrication","reason":"controlled conformance response"}'

failed = ref("PMC99900001:B1", "31665581")
absent = ref("PMC99900001:B2", "99999999")
with patch.object(ratelimit.time, "sleep", lambda *_a, **_k: None),      patch.object(band1_run.time, "sleep", lambda *_a, **_k: None):
    band1_run.process_reference(failed, must_not_call, session=FakeSession(transport_failure))
    band1_run.process_reference(absent, fabrication_fixture, session=FakeSession(healthy_absence))

failed_log, absent_log = failed.to_log_record(), absent.to_log_record()
failed_pred = failed.to_prediction().to_dict()
absent_pred = absent.to_prediction().to_dict()
failed_status = eval_report.summarize([failed_log])["f1_status"]
absent_status = eval_report.summarize([absent_log])["f1_status"]

assert failed_log["log"]["pmid_transport_status"] == FETCH_RESOLVER_ERROR
assert absent_log["log"]["pmid_transport_status"] == FETCH_ANSWERED_ABSENT
assert failed_pred["evidence"]["pmid_transport_status"] == FETCH_RESOLVER_ERROR
assert absent_pred["evidence"]["pmid_transport_status"] == FETCH_ANSWERED_ABSENT
assert failed.label != F1 and absent.label == F1
assert failed_status["transport_failed"] == 1 and failed_status["answered"] == 0
assert absent_status["transport_failed"] == 0 and absent_status["answered"] == 1

print("failed transport durable status:", failed_pred["evidence"]["pmid_transport_status"])
print("genuine absence durable status:", absent_pred["evidence"]["pmid_transport_status"])
print("failed transport Band-1 summary:", json.dumps(failed_status, sort_keys=True))
print("genuine absence Band-1 summary:", json.dumps(absent_status, sort_keys=True))
print("F1 TRANSPORT DISTINCTION THROUGH REAL SERIALIZERS: PASS")


## Cell 5 — COST GATE

Nothing below this point runs until you deliberately flip the flag.


In [ ]:
# Estimated runtime: ~0 s (hard stop before the first paid model call)
ENABLE_PAID_RUN = False
active_refs = sum(ACTIVE_REFERENCE_COUNTS.values())
print(f"Active real-paper corpus: {len(ACTIVE_PMCIDS)} paper(s), {active_refs} parsed references")
print("Cells above are free. Cells below call Anthropic for Band 1 and Band 2.")
print("The exact charge depends on extracted claim counts; this notebook does not invent a dollar estimate.")
if ENABLE_PAID_RUN is not True:
    raise RuntimeError("COST GATE CLOSED — set ENABLE_PAID_RUN = True only when ready to run the real papers")


## Cell 6 — Band 1 on real papers

Runs F1, F2, and F8, then writes the canonical lossless handoff.


In [ ]:
# Estimated runtime: minutes (Band 1 on the active real papers; model calls only for flagged survivors)
from collections import Counter
from unittest.mock import patch
from cre.f1 import run as band1_run

assert ENABLE_PAID_RUN is True
ANTHROPIC_API_KEY = get_secret("ANTHROPIC_API_KEY", required=True)

BAND1_DIR = RUN_ROOT / "band1"
BAND1_PREDICTIONS = BAND1_DIR / "band1_predictions.jsonl"
BAND1_LOGS = BAND1_DIR / "band1_logs.jsonl"
BAND1_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

if BAND1_LOGS.exists() and BAND1_PREDICTIONS.exists():
    print("Reusing complete Band-1 artifacts; no model calls repeated.")
    BAND1_MODEL_CALLS = None
    band1_logs = read_jsonl(BAND1_LOGS)
    BAND1_COUNTS = dict(Counter(r.get("label") for r in band1_logs))
else:
    assert not BAND1_LOGS.exists() and not BAND1_PREDICTIONS.exists(),         "Partial Band-1 artifacts found. They are preserved; choose a new RUN_ID instead of overwriting them."
    base_complete = band1_run.make_completer(
        MODEL, ANTHROPIC_API_KEY, max_tokens=400, max_retries=0
    )
    BAND1_MODEL_CALLS = 0
    def counted_complete(prompt):
        global BAND1_MODEL_CALLS
        BAND1_MODEL_CALLS += 1
        if BAND1_MODEL_CALLS == 1 or BAND1_MODEL_CALLS % 10 == 0:
            print(f"Band 1 model calls: {BAND1_MODEL_CALLS}", flush=True)
        return base_complete(prompt)
    with patch.object(band1_run, "make_completer", return_value=counted_complete):
        BAND1_COUNTS = band1_run.run(
            str(XML_DIR), str(BAND1_PREDICTIONS), str(BAND1_LOGS),
            model=MODEL, anthropic_key=ANTHROPIC_API_KEY,
            ncbi_key=NCBI_API_KEY, crossref_mailto=EMAIL, openalex_mailto=EMAIL
        )
    band1_logs = read_jsonl(BAND1_LOGS)

BAND1_REPORT = eval_report.summarize(band1_logs)
label_counts = dict(Counter(r.get("label") for r in band1_logs))
f1s = BAND1_REPORT["f1_status"]
resolved_rows = [r for r in band1_logs if (r.get("log") or {}).get("pmid_resolved")]
f8_answered = sum((r.get("log") or {}).get("retracted") is not None for r in resolved_rows)
snapshot = datetime.now(timezone.utc).date().isoformat()

check_attestations = {
    "F1": {"performed": True, "source": "Band-1 run.run on retrieved PMC XML",
           "snapshot_date": snapshot, "attempted": f1s["attempted"],
           "answered": f1s["answered"], "transport_failed": f1s["transport_failed"],
           "fired": label_counts.get("F1", 0)},
    "F2": {"performed": True, "source": "Band-1 deterministic identity matcher",
           "snapshot_date": snapshot, "attempted": len(band1_logs),
           "answered": len(band1_logs) - f1s["transport_failed"],
           "transport_failed": f1s["transport_failed"], "fired": label_counts.get("F2", 0)},
    "F8": {"performed": True, "source": "Band-1 PubMed publication-type lookup",
           "snapshot_date": snapshot, "attempted": len(resolved_rows),
           "answered": f8_answered, "transport_failed": len(resolved_rows) - f8_answered,
           "fired": label_counts.get("F8", 0)},
}

DISPOSITION_PATH = BAND1_DIR / preband_disposition.ARTIFACT_FILENAME
DISPOSITION_MANIFEST_PATH = Path(str(DISPOSITION_PATH) + preband_disposition.MANIFEST_SUFFIX)
if DISPOSITION_PATH.exists() or DISPOSITION_MANIFEST_PATH.exists():
    assert DISPOSITION_PATH.exists() and DISPOSITION_MANIFEST_PATH.exists(),         "Partial disposition artifacts found; choose a new RUN_ID rather than overwriting them."
    disposition_manifest = json.loads(DISPOSITION_MANIFEST_PATH.read_text(encoding="utf-8"))
else:
    disposition_manifest = preband_disposition.write_disposition(
        str(BAND1_LOGS), str(DISPOSITION_PATH), f2_commit=CODE_COMMIT,
        corpus_manifest_path=str(CORPUS_MANIFEST_PATH),
        generated_by="CRE whole-system first-light notebook",
        generated_at=datetime.now(timezone.utc).isoformat(),
        check_attestations=check_attestations,
    )

assert disposition_manifest["row_count"] == len(band1_logs)
print("Band 1 labels:", json.dumps(label_counts, sort_keys=True))
print("Band 1 model calls:", BAND1_MODEL_CALLS if BAND1_MODEL_CALLS is not None else "reused")
print("canonical disposition:", DISPOSITION_PATH)
print("BAND 1 (F1/F2/F8): PASS")


## Cell 7 — Band 2 on the same papers

Runs the real claim extraction, coverage, F3, F4, and F6 paths over Band-1-cleared references.


In [ ]:
# Estimated runtime: minutes (Band 2 on every Band-1-cleared reference)
from functools import partial
from anthropic import Anthropic

assert ENABLE_PAID_RUN is True
BAND2_OUT = RUN_ROOT / "band2"
BAND2_MANIFEST_PATH = BAND2_OUT / "judgment_run_manifest.json"

if BAND2_MANIFEST_PATH.exists():
    BAND2_MANIFEST = json.loads(BAND2_MANIFEST_PATH.read_text(encoding="utf-8"))
    assert BAND2_MANIFEST.get("status") == "complete",         "An incomplete Band-2 run is preserved. Choose a new RUN_ID; this engine cannot safely resume a partial document."
    BAND2_RECEIPT_SUMMARY = {"status": "reused complete manifest; no calls repeated"}
    print("Reusing complete Band-2 artifacts; no model calls repeated.")
else:
    if BAND2_OUT.exists() and any(BAND2_OUT.iterdir()):
        raise RuntimeError(
            "Partial Band-2 output exists and was preserved. Choose a new RUN_ID; "
            "automatic resume can duplicate reference rows in this engine."
        )
    BAND2_OUT.mkdir(parents=True, exist_ok=True)
    client = Anthropic(api_key=ANTHROPIC_API_KEY, max_retries=0, timeout=120.0)
    raw_model_call = band_prompts.make_anthropic_call(client, MODEL, max_tokens=1024)
    receipt = AdapterReceipt(model=MODEL, temperature="unsupported", assistant_prefill="unsupported")
    ncbi_session = requests.Session()
    abstract_cache = RUN_ROOT / "abstract_cache"

    def fetch_abstract(pmid):
        return evidence_reader.fetch_abstract(
            pmid, api_key=NCBI_API_KEY, email=EMAIL,
            session=ncbi_session, cache_dir=str(abstract_cache))

    def fetch_reflist(pmcid):
        return ncbi_meta.ncbi_pmc_reflist(
            pmcid, api_key=NCBI_API_KEY, email=EMAIL, session=ncbi_session)

    def resolve_pmcid(pmid):
        return fulltext_reader._live_resolve_pmcid(
            pmid, NCBI_API_KEY, EMAIL, ncbi_session)

    def pubtypes_lookup(pmid):
        return ncbi_meta.ncbi_pubtypes(
            pmid, NCBI_API_KEY, EMAIL, session=ncbi_session)

    seams = wrap_run_seams(
        receipt,
        extractor=band_prompts.make_extractor(raw_model_call),
        coverage_judge=band_prompts.make_coverage_judge(raw_model_call),
        fetch_abstract=fetch_abstract,
        fetch_reflist=fetch_reflist,
        discriminator_call_llm=raw_model_call,
        f4_verifier_call_llm=raw_model_call,
        f3_fetch_reflist=fetch_reflist,
        f3_resolve_pmcid=resolve_pmcid,
        pubtypes_lookup=pubtypes_lookup,
    )

    print("Starting Band 2 on the real-paper corpus...", flush=True)
    t0 = time.time()
    BAND2_MANIFEST = judgment_run.run_natural_judgment(
        str(XML_DIR), str(BAND2_OUT),
        preband_disposition=str(DISPOSITION_PATH),
        corpus_manifest_path=str(CORPUS_MANIFEST_PATH),
        code_commit=CODE_COMMIT, model=MODEL,
        email=EMAIL, api_key=NCBI_API_KEY, session=ncbi_session,
        f4_verifier_model_id=MODEL,
        assistant_prefill="unsupported", temperature="unsupported",
        require_full_coverage=True, require_reportable=False, production=False,
        **seams,
    )
    elapsed = time.time() - t0
    BAND2_RECEIPT_SUMMARY = receipt.summary()
    print(f"Band 2 finished in {elapsed/60:.1f} min")

assert BAND2_MANIFEST.get("status") == "complete"
print("Band 2 seam-call receipt:", json.dumps(BAND2_RECEIPT_SUMMARY, indent=2, sort_keys=True))
print("Band 2 manifest:", BAND2_MANIFEST["manifest_path"])
print("BAND 2 (F3/F4/F6 live; F5/F7 honestly unwired): PASS")


## Cell 8 — whole-system reachability report

Every count is interpreted only after its run state is derived from the artifacts.


In [ ]:
# Estimated runtime: ~3 s (artifact-derived reachability and counts)
from collections import Counter

manifest = BAND2_MANIFEST
seam_status = manifest.get("seam_status") or {}
assert set(f"F{i}" for i in range(1, 9)) <= set(seam_status), seam_status.keys()
disposition_manifest = json.loads(DISPOSITION_MANIFEST_PATH.read_text(encoding="utf-8"))
attest = disposition_manifest.get("check_attestations") or {}
preband_counts = disposition_manifest.get("label_counts") or {}
finding_counts = manifest.get("finding_labels") or {}
emitted_counts = manifest.get("emitted_labels") or {}

rows = []
for label in ("F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8"):
    if label in ("F1", "F2", "F8"):
        state = attest.get(label) or {}
        reachable = bool(state.get("performed"))
        wired = state.get("performed")
        gate = "Band-1 attestation"
        findings = int(preband_counts.get(label, 0))
        interpretation = (f"{findings} finding(s) emitted" if findings else
                          "performed; zero findings emitted" if reachable else
                          "upstream check did not run")
        if label == "F8":
            gate = "Band-1 PubMed publication-type lookup"
    else:
        state = seam_status.get(label) or {}
        wired = bool(state.get("wired"))
        gate = state.get("gate") or ""
        reachable = wired
        findings = int(finding_counts.get(label, 0))
        interpretation = (f"{findings} finding(s) emitted" if findings else
                          "reachable; zero findings emitted" if reachable else
                          "check did not run")
    if label == "F5" and not reachable:
        interpretation = "NOT RUN ON REAL DATA — no production evidence builder supplied"
    if label == "F7" and not reachable:
        interpretation = "NOT RUN — repository declares no production evidence builder"
    if label == "F6" and "cocitation.py" not in production_launcher.GOVERNING_MODULES:
        interpretation += "; WARNING: cocitation.py is not in GOVERNING_MODULES"
    rows.append((label, wired, gate, "YES" if reachable else "NO", findings, interpretation))

print("REACHABILITY COMES BEFORE COUNTS")
print("stratum | wired/performed? | gate | reachable? | findings | interpretation")
for row in rows:
    print(" | ".join(map(str, row)))

f4 = manifest.get("f4") or {}
partition = manifest.get("sentence_partition_diagnostics") or {}
print("\nF4 outcome counts:", json.dumps(f4.get("outcome_counts") or {}, sort_keys=True))
print("F4 hold reasons:", json.dumps(f4.get("hold_reason_counts") or {}, sort_keys=True))
print("sentence partition diagnostics:", json.dumps({
    "affected_reference_records": partition.get("affected_reference_records"),
    "unique_nonpartitioning_blocks": partition.get("unique_nonpartitioning_blocks"),
    "uncovered_characters": partition.get("uncovered_characters"),
}, sort_keys=True))
reportability = manifest.get("reportability") or {}
print("manifest reportability (artifact-derived, not predeclared):",
      json.dumps(reportability, indent=2, sort_keys=True)[:1800])
print("production launcher used: NO — this is a whole-system first-light run")
if reportability.get("reportable") is True and "cocitation.py" not in production_launcher.GOVERNING_MODULES:
    print("GOVERNANCE WARNING: manifest says reportable, but the F6 suppression module is outside the governed hash set. Do not treat F6 as publication-ready.")

assert all((attest.get(x) or {}).get("performed") is True for x in ("F1", "F2", "F8"))
assert seam_status["F3"]["wired"] is True
assert seam_status["F4"]["wired"] is True
assert seam_status["F6"]["wired"] is True
assert seam_status["F5"]["wired"] is False
assert seam_status["F7"]["wired"] is False
print("WHOLE-SYSTEM REACHABILITY REPORT: PASS")


## Cell 9 — inspect real decisions

Shows the actual citing sentence, cited record, claims, evidence verdicts, findings, and holds.


In [ ]:
# Estimated runtime: ~2 s (show actual paper/citation decisions)
band1_logs = read_jsonl(BAND1_LOGS)
band2_rows = read_jsonl(BAND2_OUT / "judgment_predictions.jsonl")

print("=" * 80)
print("BAND 1 — real reference outcomes")
print("=" * 80)
interesting1 = [r for r in band1_logs if r.get("label") != "cleared"] or band1_logs[:3]
for r in interesting1[:5]:
    lg = r.get("log") or {}
    claimed = r.get("claimed") or {}
    retrieved = r.get("retrieved") or {}
    print("\ncitation:", r.get("citation_id"), "| label:", r.get("label"))
    print("claimed:", str(claimed.get("title") or "")[:220])
    print("retrieved:", str(retrieved.get("title") or "")[:220])
    print("transport:", lg.get("pmid_transport_status"), "| identity:", lg.get("identity_disposition"))
    print("reason:", str(r.get("rationale") or lg.get("notes") or "")[:350])

print("\n" + "=" * 80)
print("BAND 2 — real claim/coverage/discriminator outcomes")
print("=" * 80)
interesting2 = [r for r in band2_rows if r.get("findings") or r.get("hold_reasons")]
if not interesting2:
    interesting2 = band2_rows[:3]
for r in interesting2[:5]:
    print("\ncitation:", r.get("citation_id"), "| route:", r.get("route"),
          "| findings:", r.get("findings"))
    print("sentence:", str(r.get("citing_sentence") or "")[:400])
    claims = r.get("atomic_claims") or []
    verdicts = r.get("coverage_verdicts") or []
    for i, claim in enumerate(claims[:3]):
        verdict = verdicts[i] if i < len(verdicts) else {}
        print(f"  claim {i}:", str(claim)[:240])
        print("    established:", verdict.get("established"),
              "| rationale:", str(verdict.get("rationale") or "")[:260])
    if r.get("strength_records"):
        print("  F4 strength:", json.dumps(r["strength_records"][:2], ensure_ascii=False)[:700])
    if r.get("provenance"):
        print("  F3 provenance:", json.dumps(r["provenance"], ensure_ascii=False)[:700])
    if r.get("hold_reasons"):
        print("  holds:", r.get("hold_reasons"))

print(f"\nArtifacts preserved under: {RUN_ROOT}")


## What this notebook proves—and what it does not

It proves that real PMC papers can travel through the current two-band system with one lossless ID domain and that the resulting reachability statements match the artifacts. Manifest reportability is printed exactly as computed; it is not predicted from the notebook's development label.

It does **not** claim precision, re-derive an old rate, or call an unavailable detector clean. F5 and F7 need production evidence builders before a real-paper notebook can execute them honestly. Even when the manifest reports `reportable: true`, this first-light notebook did not use `production_launcher.launch`, and F6 remains incompletely governed while `cocitation.py` is absent from `GOVERNING_MODULES`.

The default active paper is `PMC13295119` (289 parsed references in the validated retrieval). The two smaller downloaded papers remain optional smoke-test inputs, but they are not the default whole-system corpus.
